# 07주차 · 임베딩과 검색 시스템 평가

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 검색 평가 지표를 직접 계산한다.
- 모델 평균 점수와 질의별 실패 사례를 함께 분석한다.
- 평가 집합을 모델 선택과 최종 보고에 중복 사용하지 않는다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


## 순위 기반 평가

$$Precision@k=\frac{\text{상위 k개의 관련 문서 수}}{k}$$

$$MRR=\frac{1}{|Q|}\sum_q\frac{1}{rank_q}$$


In [ ]:
def precision_at_k(relevance, k):
    return sum(relevance[:k]) / k

def recall_at_k(relevance, k, total_relevant):
    return sum(relevance[:k]) / total_relevant if total_relevant else 0

def reciprocal_rank(relevance):
    return next((1 / i for i, r in enumerate(relevance, 1) if r), 0)

def ndcg_at_k(gains, k):
    gains = np.asarray(gains[:k], dtype=float)
    discounts = np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum((2**gains - 1) / discounts)
    ideal = np.sort(gains)[::-1]
    idcg = np.sum((2**ideal - 1) / discounts)
    return dcg / idcg if idcg else 0


In [ ]:
runs = {
    "TF-IDF": [1, 0, 1, 0, 0],
    "Dense": [1, 1, 0, 0, 0],
    "Hybrid": [1, 1, 0, 1, 0],
}
rows = []
for name, rel in runs.items():
    rows.append([name, precision_at_k(rel, 3), recall_at_k(rel, 3, 3), reciprocal_rank(rel), ndcg_at_k(rel, 5)])
pd.DataFrame(rows, columns=["검색기", "P@3", "R@3", "RR", "nDCG@5"]).round(3)


## 오류·편향 점검표

- 특정 지역명이나 집단명이 들어간 질의에서 성능이 일관되게 낮은가?
- 짧은 질의와 긴 질의의 성능 차이는 무엇인가?
- 평가 정답을 만든 사람 사이의 불일치는 어느 정도인가?
- 모델 선택에 사용한 질의를 최종 성능 보고에도 사용하지 않았는가?


## 학생 활동

질의 10개, 후보 문서 20개 이상으로 평가표를 만들고 두 검색기를 비교하라. 평균 점수 외에 가장 큰 실패 세 건을 제시하라.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
